# 01 — Data Preparation
**DRM Key Demand Forecasting**

This notebook loads the raw DRM key issuance data, cleans it, engineers time-series features, and exports a clean daily aggregated dataset for downstream analysis and forecasting.

---

**Pipeline:**
1. Load raw data from CSV exports (originally from Azure SQL)
2. Replicate the SQL business logic in Python (`VW_DRM_BASE`)
3. Compute daily DRM key count — `COUNT DISTINCT(CustomerID)` per day
4. Handle missing dates, outliers, and data quality issues
5. Engineer time-series features (lag, rolling, calendar)
6. Export clean dataset for EDA and modeling

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')

## 1. Load Raw Data

Data exported from Azure SQL via DataGrip as CSV files. Adjust paths if your Dataset folder structure differs.

In [ ]:
DATA_DIR = os.path.join('..', 'Dataset')

# --- Source A: Premium live channels (IPTV — K+, Đặc Sắc) ---
df_iptv = pd.read_csv(
    os.path.join(DATA_DIR, 'Log_Get_DRM_List.csv'),
    parse_dates=['Date']
)
print(f'Log_Get_DRM_List  : {df_iptv.shape[0]:>10,} rows')

# --- Source B-1: Fim+ on-demand movies ---
df_fimplus = pd.read_csv(
    os.path.join(DATA_DIR, 'Log_Fimplus_MovieID.csv'),
    parse_dates=['date']
)
print(f'Log_Fimplus_MovieID: {df_fimplus.shape[0]:>10,} rows')

# --- Source B-2: BHD on-demand movies ---
df_bhd = pd.read_csv(
    os.path.join(DATA_DIR, 'Log_BHD_MovieID.csv'),
    parse_dates=['DATE']
)
print(f'Log_BHD_MovieID   : {df_bhd.shape[0]:>10,} rows')

# --- Metadata: movie properties (isDRM flag) ---
df_movies = pd.read_csv(
    os.path.join(DATA_DIR, 'MV_PropertiesShowVN.csv')
)
print(f'MV_PropertiesShowVN: {df_movies.shape[0]:>10,} rows')

In [ ]:
print('=== Log_Get_DRM_List (IPTV) ===')
display(df_iptv.head(3))
print(df_iptv.dtypes)

print('\n=== Log_Fimplus_MovieID ===')
display(df_fimplus.head(3))
print(df_fimplus.dtypes)

print('\n=== Log_BHD_MovieID ===')
display(df_bhd.head(3))
print(df_bhd.dtypes)

print('\n=== MV_PropertiesShowVN ===')
display(df_movies.head(3))
print(df_movies.dtypes)

## 2. Replicate SQL Business Logic — Build `VW_DRM_BASE`

Apply the same logic as `01_vw_drm_base.sql`:
- Filter Fim+ and BHD logs to DRM-only movies (`isDRM = 1`)
- IPTV logs require no additional filter
- Union all three sources into a single base table

In [ ]:
drm_movie_ids = df_movies.loc[df_movies['isDRM'] == 1, 'MovieId'].unique()
print(f'DRM-protected movies: {len(drm_movie_ids)}')

In [ ]:
# --- Source A: IPTV (Premium channels) — no filter needed ---
iptv_base = df_iptv[['Date', 'CustomerID']].copy()
iptv_base.columns = ['LogDate', 'CustomerID']
iptv_base['Source'] = 'IPTV'
iptv_base['LogDate'] = pd.to_datetime(iptv_base['LogDate']).dt.normalize()

# --- Source B-1: Fim+ — filter isDRM = 1 ---
fimplus_drm = df_fimplus[df_fimplus['MovieId'].isin(drm_movie_ids)].copy()
fimplus_base = fimplus_drm[['date', 'CustomerID']].copy()
fimplus_base.columns = ['LogDate', 'CustomerID']
fimplus_base['Source'] = 'Fimplus'
fimplus_base['LogDate'] = pd.to_datetime(fimplus_base['LogDate']).dt.normalize()

# --- Source B-2: BHD — filter isDRM = 1 ---
bhd_drm = df_bhd[df_bhd['MovieID'].isin(drm_movie_ids)].copy()
bhd_base = bhd_drm[['DATE', 'CustomerID']].copy()
bhd_base.columns = ['LogDate', 'CustomerID']
bhd_base['Source'] = 'BHD'
bhd_base['LogDate'] = pd.to_datetime(bhd_base['LogDate']).dt.normalize()

# --- UNION ALL ---
df_drm_base = pd.concat([iptv_base, fimplus_base, bhd_base], ignore_index=True)

print(f'VW_DRM_BASE total records: {len(df_drm_base):,}')
print(f'Date range: {df_drm_base["LogDate"].min()} → {df_drm_base["LogDate"].max()}')
print(f'\nRecords by source:')
print(df_drm_base['Source'].value_counts())

## 3. Compute Daily DRM Key Count

Core metric: **COUNT DISTINCT(CustomerID)** per day — one key per unique customer per day regardless of how many DRM-protected items they accessed.

In [ ]:
df_daily = (
    df_drm_base
    .groupby('LogDate')['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'Total_Daily_Keys'})
    .sort_values('LogDate')
    .reset_index(drop=True)
)

print(f'Daily key count: {len(df_daily)} days')
print(f'Date range: {df_daily["LogDate"].min().date()} → {df_daily["LogDate"].max().date()}')
display(df_daily.describe())

In [ ]:
# Source-level breakdown per day (for later analysis)
df_daily_source = (
    df_drm_base
    .groupby(['LogDate', 'Source'])['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'Keys'})
    .pivot_table(index='LogDate', columns='Source', values='Keys', fill_value=0)
    .reset_index()
)

display(df_daily_source.head())

## 4. Handle Missing Dates & Data Quality

Time-series models require a continuous date index. Fill any gaps and flag anomalies.

In [ ]:
full_date_range = pd.date_range(
    start=df_daily['LogDate'].min(),
    end=df_daily['LogDate'].max(),
    freq='D'
)

missing_dates = full_date_range.difference(df_daily['LogDate'])
print(f'Missing dates in range: {len(missing_dates)}')
if len(missing_dates) > 0:
    print(missing_dates.tolist())

In [ ]:
df_daily = (
    df_daily
    .set_index('LogDate')
    .reindex(full_date_range)
    .rename_axis('LogDate')
    .reset_index()
)

# Fill missing days with 0 (no keys issued = system downtime or no access)
df_daily['Total_Daily_Keys'] = df_daily['Total_Daily_Keys'].fillna(0).astype(int)

print(f'After fill: {len(df_daily)} continuous days')
print(f'Null values: {df_daily["Total_Daily_Keys"].isnull().sum()}')

In [ ]:
# Outlier detection using IQR method
Q1 = df_daily['Total_Daily_Keys'].quantile(0.25)
Q3 = df_daily['Total_Daily_Keys'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_daily[
    (df_daily['Total_Daily_Keys'] < lower_bound) |
    (df_daily['Total_Daily_Keys'] > upper_bound)
]

print(f'IQR bounds: [{lower_bound:.0f}, {upper_bound:.0f}]')
print(f'Outlier days: {len(outliers)} ({len(outliers)/len(df_daily)*100:.1f}%)')
if len(outliers) > 0:
    display(outliers)

## 5. Feature Engineering

Create calendar features and rolling statistics that will feed into forecasting models.

In [ ]:
df_daily['DayOfWeek'] = df_daily['LogDate'].dt.dayofweek          # 0=Mon, 6=Sun
df_daily['DayOfWeekName'] = df_daily['LogDate'].dt.day_name()
df_daily['IsWeekend'] = (df_daily['DayOfWeek'] >= 5).astype(int)
df_daily['DayOfMonth'] = df_daily['LogDate'].dt.day
df_daily['WeekOfYear'] = df_daily['LogDate'].dt.isocalendar().week.astype(int)
df_daily['Month'] = df_daily['LogDate'].dt.month
df_daily['Year'] = df_daily['LogDate'].dt.year
df_daily['Quarter'] = df_daily['LogDate'].dt.quarter

# Is start/end of month (procurement-relevant)
df_daily['IsMonthStart'] = df_daily['LogDate'].dt.is_month_start.astype(int)
df_daily['IsMonthEnd'] = df_daily['LogDate'].dt.is_month_end.astype(int)

print('Calendar features added.')
df_daily.head(3)

In [ ]:
# Rolling statistics
df_daily['MA_7'] = df_daily['Total_Daily_Keys'].rolling(window=7, min_periods=1).mean()
df_daily['MA_14'] = df_daily['Total_Daily_Keys'].rolling(window=14, min_periods=1).mean()
df_daily['MA_30'] = df_daily['Total_Daily_Keys'].rolling(window=30, min_periods=1).mean()
df_daily['Std_7'] = df_daily['Total_Daily_Keys'].rolling(window=7, min_periods=1).std()

# Lag features
df_daily['Lag_1'] = df_daily['Total_Daily_Keys'].shift(1)
df_daily['Lag_7'] = df_daily['Total_Daily_Keys'].shift(7)
df_daily['Lag_14'] = df_daily['Total_Daily_Keys'].shift(14)
df_daily['Lag_30'] = df_daily['Total_Daily_Keys'].shift(30)

# Week-over-week change
df_daily['WoW_Change'] = df_daily['Total_Daily_Keys'] - df_daily['Lag_7']
df_daily['WoW_Change_Pct'] = (
    df_daily['WoW_Change'] / df_daily['Lag_7'].replace(0, np.nan) * 100
)

print('Rolling & lag features added.')
print(f'Columns: {list(df_daily.columns)}')

## 6. Data Quality Summary & Export

In [ ]:
print('=' * 60)
print('DATA PREPARATION SUMMARY')
print('=' * 60)
print(f'Date range       : {df_daily["LogDate"].min().date()} → {df_daily["LogDate"].max().date()}')
print(f'Total days       : {len(df_daily)}')
print(f'Avg daily keys   : {df_daily["Total_Daily_Keys"].mean():,.0f}')
print(f'Median daily keys: {df_daily["Total_Daily_Keys"].median():,.0f}')
print(f'Max daily keys   : {df_daily["Total_Daily_Keys"].max():,}')
print(f'Min daily keys   : {df_daily["Total_Daily_Keys"].min():,}')
print(f'Missing values   : {df_daily.isnull().sum().sum()}')
print(f'Features         : {df_daily.shape[1]} columns')
print('=' * 60)

In [ ]:
OUTPUT_DIR = os.path.join('..', 'Dataset')
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_daily.to_csv(os.path.join(OUTPUT_DIR, 'daily_drm_keys_clean.csv'), index=False)
df_daily_source.to_csv(os.path.join(OUTPUT_DIR, 'daily_drm_keys_by_source.csv'), index=False)

print('Exported:')
print(f'  → {OUTPUT_DIR}/daily_drm_keys_clean.csv      ({len(df_daily)} rows)')
print(f'  → {OUTPUT_DIR}/daily_drm_keys_by_source.csv  ({len(df_daily_source)} rows)')

In [ ]:
display(df_daily.tail(10))